# Single-field dynamic neural field simulation

This notebook demonstrates a basic 1D dynamic neural field with two competing Gaussian inputs. The field implements the Amari equation with lateral inhibition, producing self-sustaining peaks that persist after input removal.

We then extract target activations at the input positions and track the peak of the field over time. This can be used as input to a task dynamic equation, which is covered in `examples/field_to_task`.

In [ ]:
from pyphonplan import DynamicField, Targets
from pyphonplan.viz import plot_field_heatmap, plot_field_surface, plot_target_activations

## Create the field

Set up a field spanning [-10, 10] (our phonetic parameter) with a sigmoidal threshold and an interaction kernel (local excitation, lateral inhibition, global inhibition).


In [ ]:
field = DynamicField(x_min=-10, x_max=10, step_size=0.1)
field.set_sigmoid(beta=1.5, threshold=0.0)
field.set_kernel(c_exc=1, c_inh=1, c_global=0.05, sigma_exc=1.0, sigma_inh=5.0, expand=3.0)

### Sigmoid and kernel

The sigmoid gates which parts of the field contribute to self-excitation.

In [ ]:
field.plot_sigmoid();

The kernel determines the spatial pattern of lateral interactions.

In [ ]:
field.plot_kernel();

## Add inputs

Two Gaussian inputs at positions -5 and +5, with staggered timing. `input1` is active from [50, 150], `input2` from [100, 200]. Let's assume for now that these values represent milliseconds and they are integer values. Assuming integer values in the dynamic field simplifies some later analyses.

In [ ]:
field.add_input("input1", amplitude=5, position=-5, width=1.0, start=50, end=150)
field.add_input("input2", amplitude=5, position=5, width=1.0, start=100, end=200)
field.plot_inputs();

## Solve the field

Now we can solve the dynamic field equation, which integrates the above inputs and the sigmoid and interaction kernel. We add a small amount of noise (`noise=1.0`).

In [ ]:
field.solve(t_start=0, t_end=250, dt=1, tau=25.0, h=-2.0, noise=1.0)

## Visualise field activation

The heatmap shows activation over time (x-axis) and space (y-axis). The red line shows the point at which activation crosses zero, and the white dashed line tracks the spatial position corresponding to peak activation at each time step.

In [ ]:
plot_field_heatmap(field.time, field.x, field.activation);

We can also visualise as a 3D surface, where the grey plane shows the user-defined activation threshold.

In [ ]:
plot_field_surface(field.time, field.x, field.activation, threshold=0.0);

## Extract targets

In some cases, we might want to track the evolution of particular spatial values across the field. In the above example, we specified inputs at `-5.0` and `5.0`. Let's extract activation time series at the two input positions and visualise over time.

In [ ]:
targets = Targets(field, positions=[-5.0, 5.0])
plot_target_activations(field.time, targets.traces);

### Peak activation tracking

We can also track the position and activation of the field's peak over time. This can be used a time-varying input to a task dynamic equation, which is covered in `examples/field_to_task.ipynb`. We can see that peak activation follows a step function pattern (a switch from one target to another, plus a bit of noise due to the field's `noise` parameter).

In [ ]:
targets.peak_activation(plot=True);

## Animate field activation

An alternative method is to visualise the field as an activation distribution and animate how this changes over time. The following code saves a frame-by-frame animation of the 1D activation profile showing how peaks emerge and compete over time.

In [ ]:
from pyphonplan.viz import animate_field
animate_field(field.time, field.x, field.activation, save_path="single_field.mp4", show=False);